# FOMC Statement Entropy vs. VIX

This notebook visualizes the relationship between:

- **FOMC text entropy** – Shannon entropy of each post-meeting press statement (a proxy for communication *uncertainty / complexity*).
- **VIX** – CBOE Volatility Index, a forward-looking measure of market uncertainty.

We also overlay Fed Funds rate surprises and highlight the event-study results.

---
> **Note**: The data cells require the output files produced by the `data/` and `analysis/` scripts.  
> A synthetic demo data-set is generated automatically if the files are not found.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
})

## 1. Load data (or generate synthetic demo)

In [ ]:
ENTROPY_PATH  = Path('../data/fomc_entropy.csv')
VIX_PATH      = Path('../data/vix.csv')
SURPRISES_PATH = Path('../data/fed_funds_surprises.csv')
EVENT_STUDY_PATH = Path('../data/event_study_results.csv')

def synthetic_fomc_entropy(n: int = 80, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic FOMC entropy data for demo purposes."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range('2000-01-01', periods=n, freq='6W')
    # Entropy has a slight upward trend and random noise
    entropy = 7.5 + np.linspace(0, 1.5, n) + rng.normal(0, 0.3, n)
    unc_share = 0.03 + 0.01 * np.linspace(0, 1, n) + rng.normal(0, 0.005, n)
    return pd.DataFrame({'date': dates, 'entropy': entropy,
                         'uncertainty_share': np.clip(unc_share, 0, 1)})

def synthetic_vix(n_days: int = 6000, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic daily VIX data for demo purposes."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range('2000-01-03', periods=n_days)
    # Mean-reverting VIX around 20, with occasional spikes
    vix = [20.0]
    for _ in range(n_days - 1):
        shock = rng.normal(0, 0.8)
        new_val = vix[-1] + 0.05 * (20 - vix[-1]) + shock
        vix.append(max(new_val, 9.0))
    return pd.DataFrame({'date': dates, 'vix': vix})

def synthetic_surprises(fomc_dates: pd.Series, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic Fed Funds surprises."""
    rng = np.random.default_rng(seed)
    surprises = rng.normal(0, 0.05, len(fomc_dates))
    return pd.DataFrame({'fomc_date': fomc_dates.values, 'surprise': surprises})

def synthetic_event_study() -> pd.DataFrame:
    """Generate synthetic event study aggregated results."""
    rng = np.random.default_rng(42)
    t_range = range(-5, 11)
    records = []
    for t in t_range:
        mean = 0.005 * max(t, 0) + rng.normal(0, 0.002)
        se = 0.003
        records.append({
            't': t, 'mean_abnormal': mean, 'se_abnormal': se,
            'ci_lower': mean - 1.96 * se, 'ci_upper': mean + 1.96 * se,
        })
    return pd.DataFrame(records)

# --- Load real data if available, else use synthetic ---
if ENTROPY_PATH.exists():
    entropy_df = pd.read_csv(ENTROPY_PATH, parse_dates=['date'])
    print(f'Loaded real entropy data: {len(entropy_df)} rows')
else:
    entropy_df = synthetic_fomc_entropy()
    print('Using SYNTHETIC entropy data (run data/ scripts to use real data)')

if VIX_PATH.exists():
    vix_df = pd.read_csv(VIX_PATH, parse_dates=['date'])
    print(f'Loaded real VIX data: {len(vix_df)} rows')
else:
    vix_df = synthetic_vix()
    print('Using SYNTHETIC VIX data')

if SURPRISES_PATH.exists():
    surprises_df = pd.read_csv(SURPRISES_PATH, parse_dates=['fomc_date'])
    surprises_df.rename(columns={'fomc_date': 'date'}, inplace=True)
    print(f'Loaded real surprises data: {len(surprises_df)} rows')
else:
    surprises_df = synthetic_surprises(entropy_df['date'])
    surprises_df.rename(columns={'fomc_date': 'date'}, inplace=True)
    print('Using SYNTHETIC surprises data')

if EVENT_STUDY_PATH.exists():
    event_study_df = pd.read_csv(EVENT_STUDY_PATH)
    print(f'Loaded real event study results: {len(event_study_df)} rows')
else:
    event_study_df = synthetic_event_study()
    print('Using SYNTHETIC event study results')

## 2. Merge FOMC entropy with VIX (around meeting dates)

In [ ]:
vix_indexed = vix_df.set_index('date')['vix'].sort_index()

def get_vix_around(meeting_date, window=5):
    pre  = vix_indexed[vix_indexed.index <  meeting_date].iloc[-window:].mean()
    post = vix_indexed[vix_indexed.index >= meeting_date].iloc[:window].mean()
    return pd.Series({'vix_pre': pre, 'vix_post': post, 'vix_change': post - pre})

vix_around = entropy_df['date'].apply(get_vix_around)
merged = pd.concat([entropy_df.set_index('date'), vix_around], axis=1).reset_index()
merged = merged.merge(surprises_df[['date', 'surprise']], on='date', how='left')
merged.dropna(subset=['vix_post', 'entropy'], inplace=True)
print(f'Merged panel: {len(merged)} FOMC meetings')
merged.head()

## 3. Time Series: FOMC Entropy and VIX

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# --- Top panel: FOMC statement entropy ---
ax1.plot(merged['date'], merged['entropy'], color='steelblue', lw=1.5,
         marker='o', ms=3, label='Statement Entropy (bits)')
ax1.set_ylabel('Shannon Entropy (bits)')
ax1.set_title('FOMC Statement Entropy over Time')
ax1.legend(loc='upper left')
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

# --- Bottom panel: VIX ---
ax2.fill_between(vix_df['date'], vix_df['vix'], alpha=0.3, color='tomato')
ax2.plot(vix_df['date'], vix_df['vix'], color='tomato', lw=1, label='VIX (daily close)')
# Mark FOMC dates
for d in merged['date']:
    ax2.axvline(d, color='steelblue', alpha=0.15, lw=0.8)
ax2.set_ylabel('VIX')
ax2.set_xlabel('Date')
ax2.set_title('CBOE VIX with FOMC Meeting Dates (vertical lines)')
ax2.legend(loc='upper right')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('../notebooks/fig_entropy_vix_timeseries.png', bbox_inches='tight')
plt.show()

## 4. Scatter: FOMC Entropy vs. VIX (post-meeting)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (y_col, y_label) in zip(axes,
    [('vix_post', 'Average VIX (post-meeting, 5 days)'),
     ('vix_change', 'VIX Change Around FOMC (post − pre)')]):

    x = merged['entropy']
    y = merged[y_col]

    # Scatter
    ax.scatter(x, y, alpha=0.6, edgecolors='white', linewidth=0.3, s=50)

    # OLS regression line
    slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
    x_fit = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_fit, intercept + slope * x_fit, color='tomato', lw=2,
            label=f'OLS: slope={slope:.2f}, R²={r_val**2:.3f} (p={p_val:.3f})')

    ax.set_xlabel('Shannon Entropy of FOMC Statement (bits)')
    ax.set_ylabel(y_label)
    ax.set_title(f'Entropy vs. {y_label}')
    ax.legend(fontsize=9)

plt.suptitle('FOMC Communication Uncertainty and Market Volatility', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../notebooks/fig_entropy_vs_vix_scatter.png', bbox_inches='tight')
plt.show()

## 5. Uncertainty Share vs. VIX

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = merged['uncertainty_share'] * 100  # percent
y = merged['vix_post']

scatter = ax.scatter(x, y, c=merged['date'].astype(np.int64),
                     cmap='viridis', alpha=0.7, edgecolors='white',
                     linewidth=0.4, s=60)
cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Year', fontsize=9)
# Fix colorbar tick labels to show years
cbar_ticks = cbar.get_ticks()
cbar.set_ticklabels(
    [pd.Timestamp(int(t)).year if t > 0 else '' for t in cbar_ticks], fontsize=8
)

slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
x_fit = np.linspace(x.min(), x.max(), 200)
ax.plot(x_fit, intercept + slope * x_fit, 'r--', lw=2,
        label=f'OLS: slope={slope:.1f}, R²={r_val**2:.3f}')

ax.set_xlabel('Loughran-McDonald Uncertainty Words (%)')
ax.set_ylabel('Average VIX (5 days post-FOMC)')
ax.set_title('FOMC Uncertainty Vocabulary vs. Post-meeting VIX')
ax.legend()
plt.tight_layout()
plt.savefig('../notebooks/fig_unc_share_vs_vix.png', bbox_inches='tight')
plt.show()

## 6. Event Study: Cumulative Abnormal VIX Change

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

t = event_study_df['t']
mean = event_study_df['mean_abnormal']
ci_lo = event_study_df['ci_lower']
ci_hi = event_study_df['ci_upper']

ax.bar(t, mean, color=np.where(mean >= 0, 'steelblue', 'tomato'),
       alpha=0.7, width=0.8, label='Mean Abnormal ΔlnVIX')
ax.fill_between(t, ci_lo, ci_hi, alpha=0.25, color='steelblue', label='95% CI')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(0, color='darkred', lw=1.2, ls=':', label='FOMC Meeting (t=0)')

ax.set_xlabel('Event Time (trading days relative to FOMC meeting)')
ax.set_ylabel('Mean Abnormal Δln(VIX)')
ax.set_title('Event Study: Abnormal VIX Change around FOMC Meetings')
ax.legend()
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig('../notebooks/fig_event_study.png', bbox_inches='tight')
plt.show()

## 7. Joint Density: Entropy Quantiles vs. VIX Response

In [ ]:
merged['entropy_q'] = pd.qcut(merged['entropy'], q=4,
                               labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=merged, x='entropy_q', y='vix_post',
    palette='Blues', ax=ax,
    order=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'],
)
ax.set_xlabel('FOMC Entropy Quartile')
ax.set_ylabel('Average VIX (5 days post-meeting)')
ax.set_title('Post-meeting VIX by FOMC Statement Entropy Quartile')
plt.tight_layout()
plt.savefig('../notebooks/fig_entropy_quartile_vix.png', bbox_inches='tight')
plt.show()

## 8. Summary Statistics

In [ ]:
summary_cols = ['entropy', 'uncertainty_share', 'vix_pre', 'vix_post', 'vix_change']
if 'surprise' in merged.columns:
    summary_cols.append('surprise')

summary = merged[summary_cols].describe().T
summary.columns = ['N', 'Mean', 'Std', 'Min', '25%', 'Median', '75%', 'Max']
summary['N'] = summary['N'].astype(int)
print('=== Summary Statistics ===')
print(summary.round(4).to_string())

## 9. Correlation Table

In [ ]:
corr_cols = ['entropy', 'uncertainty_share', 'vix_post', 'vix_change']
if 'surprise' in merged.columns:
    corr_cols.append('surprise')

corr = merged[corr_cols].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Pearson Correlation Matrix')
plt.tight_layout()
plt.savefig('../notebooks/fig_correlation_matrix.png', bbox_inches='tight')
plt.show()
print(corr.round(3).to_string())